# DateComparator Demo

This notebook walks through `DateComparator`, stickler's deterministic, non-LLM date comparator. It parses both sides of a comparison into `datetime` objects (or date ranges) and scores them with semantic awareness, so that surface-form differences (separators, padding, year format, named months, weekday prefixes) do not penalize a model for being correct.

## What we'll cover
- Surface-form normalization (one date written many ways, all score 1.0)
- Partial dates and the `allow_partial_year` knob
- Date ranges and the `range_mode` knob (`graded`, `contains`, `strict`, `reject`)
- Two-digit year handling
- Locale resolution via `dayfirst` (US vs EU layout)
- Edge cases: corrupted input, time-only strings, ambiguous pairs
- A real-world `StructuredModel` example with two date fields
- A configuration cheat-sheet and empirical motivation

## 1. Setup and Imports

In [1]:
import os
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from datetime import timedelta

import pandas as pd
from stickler.comparators.date import DateComparator
from stickler.structured_object_evaluator.models.comparable_field import ComparableField
from stickler.structured_object_evaluator.models.structured_model import StructuredModel
print("Imports successful.")

Imports successful.


## 2. Surface-form normalization

The same calendar day can be written many ways. `DateComparator` parses both sides with `python-dateutil` and returns 1.0 whenever they resolve to the same day, regardless of separators, zero-padding, year format, named months, weekday prefixes, or trailing punctuation.

In [2]:
comparator = DateComparator()

surface_form_pairs = [
    ("10/24/2016", "10/24/16", "two-digit year"),
    ("10/24/2016", "10-24-2016", "dash separator"),
    ("10/24/2016", "10.24.2016", "dot separator"),
    ("2/1/2016", "02/01/2016", "zero-padding"),
    ("2/1/2016", "02/01/16", "zero-pad + 2-digit year"),
    ("10/24/2016", "October 24, 2016", "long month name"),
    ("10/24/2016", "Oct 24 2016", "abbrev. month, no comma"),
    ("10/24/2016", "Oct. 24, 2016", "abbrev. with period"),
    ("10/24/2016", "2016-10-24", "ISO 8601 layout"),
    ("Mon 10/24/16", "10/24/16", "weekday prefix stripped"),
    ("Mon 10/24/16", "Monday October 24, 2016", "long weekday + textual"),
    ("2025-01-01", "Jan 1, 2025", "ISO vs named month"),
]

rows = [
    {"gt": gt, "pred": pred, "variation": variation, "score": comparator.compare(gt, pred)}
    for gt, pred, variation in surface_form_pairs
]
print(pd.DataFrame(rows).to_string(index=False))

          gt                    pred               variation  score
  10/24/2016                10/24/16          two-digit year    1.0
  10/24/2016              10-24-2016          dash separator    1.0
  10/24/2016              10.24.2016           dot separator    1.0
    2/1/2016              02/01/2016            zero-padding    1.0
    2/1/2016                02/01/16 zero-pad + 2-digit year    1.0
  10/24/2016        October 24, 2016         long month name    1.0
  10/24/2016             Oct 24 2016 abbrev. month, no comma    1.0
  10/24/2016           Oct. 24, 2016     abbrev. with period    1.0
  10/24/2016              2016-10-24         ISO 8601 layout    1.0
Mon 10/24/16                10/24/16 weekday prefix stripped    1.0
Mon 10/24/16 Monday October 24, 2016  long weekday + textual    1.0
  2025-01-01             Jan 1, 2025      ISO vs named month    1.0


Every pair scores 1.0 because the parser collapses surface form before comparing.

### Tolerance for off-by-one days

`tolerance` accepts a `timedelta` (or a numeric value in days). It applies only to Tier 1 same-day comparisons (both sides year-bearing singles) and lets near-misses still score 1.0.

In [3]:
default_cmp = DateComparator()
tolerant_cmp = DateComparator(tolerance=timedelta(days=1))
week_tolerant_cmp = DateComparator(tolerance=7)  # numeric is interpreted as days

tolerance_rows = [
    {
        "gt": "2025-01-01",
        "pred": "2025-01-02",
        "default": default_cmp.compare("2025-01-01", "2025-01-02"),
        "tolerance=1d": tolerant_cmp.compare("2025-01-01", "2025-01-02"),
        "tolerance=7d": week_tolerant_cmp.compare("2025-01-01", "2025-01-02"),
    },
    {
        "gt": "2025-01-01",
        "pred": "2025-01-08",
        "default": default_cmp.compare("2025-01-01", "2025-01-08"),
        "tolerance=1d": tolerant_cmp.compare("2025-01-01", "2025-01-08"),
        "tolerance=7d": week_tolerant_cmp.compare("2025-01-01", "2025-01-08"),
    },
]
print(pd.DataFrame(tolerance_rows).to_string(index=False))

        gt       pred  default  tolerance=1d  tolerance=7d
2025-01-01 2025-01-02      0.0           1.0           1.0
2025-01-01 2025-01-08      0.0           0.0           1.0


## 3. Partial dates (year-presence)

Real extracted documents often omit the year. `DateComparator` distinguishes three cases:

- **Both sides year-less**, month/day match → 1.0 (no year was claimed on either side)
- **One side year-less, other year-bearing**, month/day match → controlled by `allow_partial_year`:
    - `False` (default): 0.0 — refuse to invent a year
    - `True`: 0.7 — partial credit for matching m/d
- **Month/day differ** → 0.0 regardless

In [4]:
default_cmp = DateComparator()  # allow_partial_year=False by default
partial_cmp = DateComparator(allow_partial_year=True)

partial_rows = [
    {
        "gt": "Oct 24",
        "pred": "10/24",
        "case": "both year-less, m/d match",
        "default": default_cmp.compare("Oct 24", "10/24"),
        "allow_partial_year=True": partial_cmp.compare("Oct 24", "10/24"),
    },
    {
        "gt": "11/03",
        "pred": "11/03/2012",
        "case": "year hallucination (179x in eval set)",
        "default": default_cmp.compare("11/03", "11/03/2012"),
        "allow_partial_year=True": partial_cmp.compare("11/03", "11/03/2012"),
    },
    {
        "gt": "Jan 1, 2024",
        "pred": "Jan 1",
        "case": "GT has year, pred year-less",
        "default": default_cmp.compare("Jan 1, 2024", "Jan 1"),
        "allow_partial_year=True": partial_cmp.compare("Jan 1, 2024", "Jan 1"),
    },
    {
        "gt": "Oct 24",
        "pred": "10/25/16",
        "case": "year-presence diff AND m/d differ",
        "default": default_cmp.compare("Oct 24", "10/25/16"),
        "allow_partial_year=True": partial_cmp.compare("Oct 24", "10/25/16"),
    },
]
print(pd.DataFrame(partial_rows).to_string(index=False))

         gt       pred                                  case  default  allow_partial_year=True
     Oct 24      10/24             both year-less, m/d match      1.0                      1.0
      11/03 11/03/2012 year hallucination (179x in eval set)      0.0                      0.7
Jan 1, 2024      Jan 1           GT has year, pred year-less      0.0                      0.7
     Oct 24   10/25/16     year-presence diff AND m/d differ      0.0                      0.0


Note that `allow_partial_year` does **not** affect the year-less-vs-year-less case — that's already 1.0 because no year was claimed on either side. The 0.7 partial-credit value is baked in (the multiplier `_PARTIAL_YEAR_MULTIPLIER`); tune the cutoff at the field level via `ComparableField.threshold` if you want partial matches to count as misses.

## 4. Date ranges

Ground truth and predictions can both be ranges (`10/24/16 to 10/30/16`), or one side can be a range while the other is a single date. The `range_mode` knob picks the scoring policy:

| Mode | range vs range | range vs single |
|---|---|---|
| `strict` | 1.0 if endpoints exact, else 0.0 | 0.0 (shape mismatch) |
| `contains` | 1.0 if endpoints exact, else 0.0 | 1.0 if single inside range, else 0.0 |
| `graded` (default) | Jaccard overlap (intersection days / union days) | 0.5 if single inside range, else 0.0 |
| `reject` | 0.0 always | 0.0 always |

Delimiters recognized are `" to "`, `" through "`, and `" - "` (spaces required around the bare dash so ISO dates like `2025-01-01` aren't shredded).

In [5]:
graded_cmp = DateComparator()                      # range_mode='graded' (default)
contains_cmp = DateComparator(range_mode="contains")
strict_cmp = DateComparator(range_mode="strict")
reject_cmp = DateComparator(range_mode="reject")

# Note: textual months ("Jan 6, 2024") are used here so the math is unambiguous
# under dayfirst=None. With purely numeric dates like "1/6/2024", dayfirst=None
# tries both interpretations and keeps the higher score, which can make the
# Jaccard math harder to reason about in a teaching example.
range_cases = [
    ("10/28/2016", "10/24/2016 to 10/30/2016", "single inside range"),
    ("10/15/2016", "10/24/2016 to 10/30/2016", "single outside range"),
    (
        "Oct 24, 2016 to Oct 30, 2016",
        "10/24/2016 - 10/30/2016",
        "range vs range, exact (mixed surface form)",
    ),
    (
        "Oct 24, 2016 to Oct 30, 2016",
        "Oct 24, 2016 to Oct 31, 2016",
        "range vs range, 7d overlap / 8d union",
    ),
    (
        "Jan 1, 2024 to Jan 10, 2024",
        "Jan 6, 2024 to Jan 15, 2024",
        "range vs range, 5d overlap / 15d union",
    ),
    (
        "Oct 28, 2016 to Oct 28, 2016",
        "Oct 28, 2016",
        "single-day range collapses to single",
    ),
]

range_rows = [
    {
        "case": case,
        "graded": round(graded_cmp.compare(gt, pred), 4),
        "contains": round(contains_cmp.compare(gt, pred), 4),
        "strict": round(strict_cmp.compare(gt, pred), 4),
        "reject": round(reject_cmp.compare(gt, pred), 4),
    }
    for gt, pred, case in range_cases
]
print(pd.DataFrame(range_rows).to_string(index=False))

                                      case  graded  contains  strict  reject
                       single inside range  0.5000       1.0     0.0     0.0
                      single outside range  0.0000       0.0     0.0     0.0
range vs range, exact (mixed surface form)  1.0000       1.0     1.0     0.0
     range vs range, 7d overlap / 8d union  0.8750       0.0     0.0     0.0
    range vs range, 5d overlap / 15d union  0.3333       0.0     0.0     0.0
      single-day range collapses to single  1.0000       1.0     1.0     0.0


A few things to notice:

- Surface-form normalization still runs inside ranges: `Oct 24, 2016 to Oct 30, 2016` matches `10/24/2016 - 10/30/2016` even with different delimiters and year formats.
- The graded Jaccard score for the 5d/15d row is `5/15 ≈ 0.333` — inclusive-day intersection (`Jan 6 – Jan 10`, 5 days) divided by inclusive-day union (`Jan 1 – Jan 15`, 15 days).
- Single-day ranges (`X to X`) collapse to a bare single under `graded`/`contains`/`strict`, but stay as ranges under `reject` so they surface as a structural mismatch.

### Range + partial year

If one side is year-less and the other is year-bearing, the base range score is multiplied by 0.7 when `allow_partial_year=True`, and zeroed otherwise.

In [6]:
default_cmp = DateComparator()  # graded, allow_partial_year=False
partial_cmp = DateComparator(allow_partial_year=True, range_mode="graded")
partial_contains_cmp = DateComparator(allow_partial_year=True, range_mode="contains")

year_mismatch_rows = [
    {
        "gt": "Oct 28",
        "pred": "10/24/16 to 10/30/16",
        "default (graded)": default_cmp.compare("Oct 28", "10/24/16 to 10/30/16"),
        "graded + allow_partial_year": partial_cmp.compare("Oct 28", "10/24/16 to 10/30/16"),
        "contains + allow_partial_year": partial_contains_cmp.compare("Oct 28", "10/24/16 to 10/30/16"),
    },
]
print(pd.DataFrame(year_mismatch_rows).to_string(index=False))

    gt                 pred  default (graded)  graded + allow_partial_year  contains + allow_partial_year
Oct 28 10/24/16 to 10/30/16               0.0                         0.35                            0.7


Under `graded`, base score 0.5 × 0.7 multiplier = 0.35. Under `contains`, base 1.0 × 0.7 = 0.7.

## 5. Two-digit year handling

`DateComparator` defers to `python-dateutil` for two-digit-year resolution. `dateutil` uses a sliding 50-year window centred on the current year, so a two-digit input is mapped to the closer of the two candidate centuries. This keeps recent years (`'24'`, `'25'`) interpreted as current-century, while clearly historical years (`'95'`) interpret as previous-century. The pivot is not configurable on this comparator — write the year out as four digits if you need exact control.

In [7]:
two_digit_cmp = DateComparator()

# Each row pairs a two-digit input against the four-digit form that dateutil
# resolves it to. Ambiguous years near the 50-year-window boundary are
# excluded because their resolution shifts as the calendar year advances.
two_digit_rows = [
    ("5/3/95", "5/3/1995", "95 -> 1995 (clearly historical)"),
    ("10/24/16", "10/24/2016", "16 -> 2016 (recent)"),
    ("5/3/24", "5/3/2024", "24 -> 2024 (recent)"),
]
rows = [
    {"gt": gt, "pred": pred, "note": note, "score": two_digit_cmp.compare(gt, pred)}
    for gt, pred, note in two_digit_rows
]
print(pd.DataFrame(rows).to_string(index=False))

print()
print("Counter-example: forcing the 'wrong' century with an explicit four-digit year")
print(f"  '5/3/95' vs '5/3/2095' = {two_digit_cmp.compare('5/3/95', '5/3/2095')}")
print(f"  '10/24/16' vs '10/24/1916' = {two_digit_cmp.compare('10/24/16', '10/24/1916')}")

      gt       pred                            note  score
  5/3/95   5/3/1995 95 -> 1995 (clearly historical)    1.0
10/24/16 10/24/2016             16 -> 2016 (recent)    1.0
  5/3/24   5/3/2024             24 -> 2024 (recent)    1.0

Counter-example: forcing the 'wrong' century with an explicit four-digit year
  '5/3/95' vs '5/3/2095' = 0.0
  '10/24/16' vs '10/24/1916' = 0.0


## 6. Locale resolution: `dayfirst`

Numeric dates like `5/3/2025` are genuinely ambiguous: is it May 3 (US) or 3 May (EU)? `dayfirst` controls how that's resolved.

- `None` (default) — try both interpretations and keep whichever gives the higher score. Pairs that disagree under both interpretations score 0.0.
- `True` — force day-first parsing (EU layout).
- `False` — force month-first parsing (US layout).

When at least one of the day/month positions is `> 12`, the layout is unambiguous and parsing succeeds regardless of `dayfirst`.

In [8]:
auto_cmp = DateComparator()                # dayfirst=None
eu_cmp = DateComparator(dayfirst=True)     # day-first
us_cmp = DateComparator(dayfirst=False)    # month-first

locale_cases = [
    ("5/3/2025", "May 3, 2025", "unambiguous textual pins US reading"),
    ("5/3/2025", "March 5, 2025", "unambiguous textual pins EU reading"),
    ("5/3/2025", "3/5/2025", "both ambiguous, no consistent reading"),
    ("13/5/2025", "5/13/2025", "day=13 forces unambiguous parse on both sides"),
]
rows = [
    {
        "gt": gt,
        "pred": pred,
        "case": case,
        "auto (None)": auto_cmp.compare(gt, pred),
        "eu (dayfirst=True)": eu_cmp.compare(gt, pred),
        "us (dayfirst=False)": us_cmp.compare(gt, pred),
    }
    for gt, pred, case in locale_cases
]
print(pd.DataFrame(rows).to_string(index=False))

       gt          pred                                          case  auto (None)  eu (dayfirst=True)  us (dayfirst=False)
 5/3/2025   May 3, 2025           unambiguous textual pins US reading          1.0                 0.0                  1.0
 5/3/2025 March 5, 2025           unambiguous textual pins EU reading          1.0                 1.0                  0.0
 5/3/2025      3/5/2025         both ambiguous, no consistent reading          0.0                 0.0                  0.0
13/5/2025     5/13/2025 day=13 forces unambiguous parse on both sides          1.0                 1.0                  1.0


Use `dayfirst=None` for evaluation pipelines that mix sources. If you know your data is consistently EU or US, pinning the value avoids mistakenly matching across layouts.

## 7. Edge cases the comparator rejects

The comparator deliberately does not auto-repair malformed inputs. The following inputs all fail to parse and return 0.0 — they surface as misses rather than silent passes. (There is no `warn_on_corrupted_input` knob in the current implementation; corrupted parses simply score 0.0.)

In [9]:
edge_cmp = DateComparator()

edge_cases = [
    ("10/24/16", "07/17/ 6", "embedded space inside year (real GT bug)"),
    ("11/03/16", "11/0316", "missing separator (real GT bug)"),
    ("10/24/16", "10/45AM", "time-only string in date field"),
    ("10/24/16", "12:30 PM", "time-only string"),
    ("10/24/16", "", "empty prediction"),
    ("10/24/16", "not a date", "unparseable text"),
]
rows = [
    {"gt": gt, "pred": pred, "reason": reason, "score": edge_cmp.compare(gt, pred)}
    for gt, pred, reason in edge_cases
]
print(pd.DataFrame(rows).to_string(index=False))

print()
print("Symmetric None handling:")
print(f"  compare(None, None) = {edge_cmp.compare(None, None)}  (both empty -> trivially equal)")
print(f"  compare(None, '10/24/16') = {edge_cmp.compare(None, '10/24/16')}  (one side missing -> 0.0)")

      gt       pred                                   reason  score
10/24/16   07/17/ 6 embedded space inside year (real GT bug)    0.0
11/03/16    11/0316          missing separator (real GT bug)    0.0
10/24/16    10/45AM           time-only string in date field    0.0
10/24/16   12:30 PM                         time-only string    0.0
10/24/16                                    empty prediction    0.0
10/24/16 not a date                         unparseable text    0.0

Symmetric None handling:
  compare(None, None) = 1.0  (both empty -> trivially equal)
  compare(None, '10/24/16') = 0.0  (one side missing -> 0.0)


## 8. Real-world example: `InvoiceRecord` with two date fields

Now let's wire `DateComparator` into a `StructuredModel` so it participates in overall scoring alongside other fields. We model an invoice with both an `invoice_date` (always present, always year-bearing) and a `due_date` (sometimes year-less in scanned documents).

In [10]:
class InvoiceRecord(StructuredModel):
    """Invoice with two date fields, each tuned for its own situation."""

    invoice_id: str = ComparableField(weight=1.0)

    # Strict date matching for the invoice date.
    invoice_date: str = ComparableField(
        comparator=DateComparator(),
        weight=2.0,
    )

    # Due dates often drop the year on paper forms; allow partial credit.
    due_date: str = ComparableField(
        comparator=DateComparator(allow_partial_year=True),
        weight=1.0,
    )

    # Optional service period reads as a range.
    service_period: str = ComparableField(
        comparator=DateComparator(range_mode="contains"),
        weight=1.0,
    )


gt = InvoiceRecord(
    invoice_id="INV-001",
    invoice_date="10/24/2016",
    due_date="11/24",
    service_period="10/01/2016 to 10/31/2016",
)

pred = InvoiceRecord(
    invoice_id="INV-001",
    invoice_date="Oct 24, 2016",        # surface variation, scores 1.0
    due_date="11/24/2016",              # year hallucination, allowed -> 0.7
    service_period="10/15/2016",        # single inside range, contains -> 1.0
)

result = gt.compare_with(pred)

print(f"Overall score: {result['overall_score']:.3f}")
print("\nField scores:")
for field, score in result["field_scores"].items():
    print(f"  {field:18}: {score:.3f}")

Overall score: 0.940

Field scores:
  invoice_id        : 1.000
  invoice_date      : 1.000
  due_date          : 0.700
  service_period    : 1.000


Per-field, the `invoice_date` matches exactly (Tier 1 surface-form), the `due_date` scores 0.7 because the prediction added a year (Tier 3 with `allow_partial_year=True`), and `service_period` scores 1.0 because the predicted single date falls inside the GT range under `range_mode="contains"`. The overall score is the weighted average of these per-field scores.

## 8b. JSON-Schema configuration (IDP accelerator)

In production IDP pipelines, models are typically declared as JSON Schema documents rather than Python classes. `DateComparator` is fully configurable via the `x-aws-stickler-comparator` and `x-aws-stickler-comparator-config` extensions — every constructor option (`tolerance`, `dayfirst`, `allow_partial_year`, `range_mode`, `threshold`) survives a JSON round-trip.

The cell below builds the same `Invoice` model from Section 8 entirely from a JSON schema string, no Python class definition required.

In [11]:
import json

invoice_schema = json.loads(
    """
    {
      "type": "object",
      "x-aws-stickler-model-name": "InvoiceFromSchema",
      "x-aws-stickler-match-threshold": 0.8,
      "properties": {
        "invoice_id": {
          "type": "string",
          "x-aws-stickler-weight": 1.0
        },
        "invoice_date": {
          "type": "string",
          "x-aws-stickler-comparator": "DateComparator",
          "x-aws-stickler-comparator-config": { "dayfirst": false },
          "x-aws-stickler-weight": 2.0
        },
        "due_date": {
          "type": "string",
          "x-aws-stickler-comparator": "DateComparator",
          "x-aws-stickler-comparator-config": { "allow_partial_year": true },
          "x-aws-stickler-weight": 1.0
        },
        "service_period": {
          "type": "string",
          "x-aws-stickler-comparator": "DateComparator",
          "x-aws-stickler-comparator-config": { "range_mode": "contains" },
          "x-aws-stickler-weight": 1.0
        }
      },
      "required": ["invoice_id"]
    }
    """
)

InvoiceFromSchema = StructuredModel.from_json_schema(invoice_schema)

gt = InvoiceFromSchema(
    invoice_id="INV-001",
    invoice_date="10/24/2016",
    due_date="11/24",
    service_period="10/01/2016 to 10/31/2016",
)
pred = InvoiceFromSchema(
    invoice_id="INV-001",
    invoice_date="Oct 24, 2016",
    due_date="11/24/2016",
    service_period="10/15/2016",
)

result = gt.compare_with(pred)
print(f"Class built from schema: {InvoiceFromSchema.__name__}")
print(f"Overall score: {result['overall_score']:.3f}")
print()
print("Field scores:")
for field, score in result["field_scores"].items():
    print(f"  {field:18}: {score:.3f}")

Class built from schema: InvoiceFromSchema
Overall score: 0.940

Field scores:
  invoice_id        : 1.000
  invoice_date      : 1.000
  due_date          : 0.700
  service_period    : 1.000


Same overall score, same per-field scores — the schema-driven flow produces the identical model. Two practical notes for IDP authors:

- Use lower-case JSON literals for booleans (`true`/`false`, not `True`/`False`) and for `null` — these are validated when the schema is parsed.
- The comparator-config keys (`dayfirst`, `allow_partial_year`, `range_mode`, `tolerance`, `threshold`) are validated by `DateComparator`'s constructor, so an invalid `range_mode` like `"fuzzy"` fails at schema-load time with a clear field-attributed error rather than silently accepting it.

## 9. Configuration cheat-sheet

All constructor knobs in one place:

| Option | Type | Default | Effect |
|---|---|---|---|
| `threshold` | `float` | `1.0` | Forwarded to `BaseComparator`; `ComparableField` uses this when deciding what counts as a match. |
| `tolerance` | `timedelta`, `int`, or `float` (days) | `timedelta(0)` | Window for Tier 1 same-day comparisons (both sides year-bearing singles). Ignored for year-less and range branches. Numeric values are interpreted as days. |
| `dayfirst` | `Optional[bool]` | `None` | `None` tries both interpretations and keeps the higher score. `True` forces day-first (EU). `False` forces month-first (US). |
| `allow_partial_year` | `bool` | `False` | When `True`, year-less ↔ year-bearing pairs with matching m/d score 0.7 (Tier 3). Also multiplies range scores by 0.7 when year-presence differs between sides. |
| `range_mode` | `'strict' \| 'reject' \| 'contains' \| 'graded'` | `'graded'` | Picks the scoring policy for range comparisons (see Section 4). |

The partial-credit multiplier (0.7) and the graded-contains base (0.5) are baked-in constants. To adjust where partial matches start counting, tune `ComparableField.threshold` rather than the comparator.

## 10. Empirical grounding

`DateComparator` was designed against a deep characterization of real document-extraction failures: 6,630 date-field failures across 14,530 comparisons from two evaluation runs (qwen3.5-2b and qwen3.6-27b on FCC invoices). The headline failure modes the comparator is built to address:

- **Year hallucination — ~1500 failures.** Predictions consistently add a year when the ground truth has none (e.g. `gt='11/03'` vs `pred='11/03/2012'` recurring 179 times verbatim). `allow_partial_year=True` lets these score 0.7 instead of 0.0.
- **Pure surface-form differences — ~400 failures.** Zero-padding, separator differences, year format differences. These score 1.0 by default — no configuration required.
- **Range-vs-range and single-vs-range — ~600+ failures.** Ranges are not an out-of-scope oddity; range-vs-range was the fifth most common failure mode (375 occurrences). The four `range_mode` policies cover the policy spectrum from strict shape-matching to Jaccard overlap.
- **Out-of-domain predictions** (`'10/45AM'`, `'9/5AM'`) and **GT data quality issues** (`'07/17/ 6'`, `'11/0316'`) are surfaced as 0.0 rather than auto-repaired.

## 11. Key takeaways

**Use `DateComparator` whenever a field holds a date or date range** — not a `LevenshteinComparator` on the string. The comparator is deterministic, fast, has no LLM dependency, and treats dates as semantic values rather than character sequences.

**Tune three knobs in this order:**
1. `range_mode` — match it to your data shape. Use `graded` (default) for evaluation that rewards partial overlap, `contains` for annotator-vs-source artifacts, `strict` for shape-sensitive evaluation, `reject` if any range on either side should be a hard miss.
2. `allow_partial_year` — turn on when your predictions or ground truth sometimes omit the year. Off by default to keep partial matches opt-in.
3. `dayfirst` — leave at `None` for mixed-locale data. Pin to `True` (EU) or `False` (US) when you know your sources.

**Think in tiers when reading a score:**
- 1.0 = same calendar day after surface-form normalization, or both year-less with matching m/d, or a same-day endpoint range match
- 0.7 = year-presence partial credit (only when `allow_partial_year=True`)
- 0.5 = single date contained in the other side's range (only under `range_mode='graded'`)
- between 0 and 1 = Jaccard overlap of two ranges (only under `range_mode='graded'`)
- 0.0 = unparseable, time-only, structurally rejected, or genuinely different dates

**Tune the cutoff at the field, not the comparator.** Set `ComparableField.threshold` to decide whether 0.7 counts as a match in your evaluation.